In [2]:
import torch
import numpy as np
import random
from minicons import scorer

MODEL = "Qwen/Qwen3-VL-2B-Instruct"
CACHE_DIR = "/mnt/dv/wid/projects3/Rogers-muri-human-ai/zstuddiford"
SEED = 0

# --- seed everything ---
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- load model ---
lm = scorer.VLMScorer(MODEL, device="cuda", torch_dtype=torch.bfloat16, cache_dir=CACHE_DIR)

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [3]:
import random, pandas as pd, re, uuid
random.seed(0)

# ============================================================
#  Correct pluralization: explicit irregular tables + rules.
# ============================================================
IRREG_PL = {
    'mouse':'mice','goose':'geese','ox':'oxen','louse':'lice',
    'sheep':'sheep','deer':'deer','fish':'fish','bison':'bison','moose':'moose',
    'cod':'cod','trout':'trout','salmon':'salmon','carp':'carp','perch':'perch',
    'pike':'pike','bass':'bass','elk':'elk','swine':'swine','gnu':'gnu',
    'aircraft':'aircraft','offspring':'offspring',
}
F_TO_VES = {
    'wolf':'wolves','calf':'calves','half':'halves','loaf':'loaves','leaf':'leaves',
    'hoof':'hooves','shelf':'shelves','thief':'thieves','wharf':'wharves',
    'dwarf':'dwarves','scarf':'scarves','elf':'elves','self':'selves',
    'knife':'knives','life':'lives','wife':'wives',
}
def pluralize(w):
    if w in IRREG_PL: return IRREG_PL[w]
    if w in F_TO_VES: return F_TO_VES[w]
    if w.endswith(('s','x','z','sh','ch')):           return w + 'es'
    if w.endswith('y') and len(w) > 1 and w[-2] not in 'aeiou': return w[:-1] + 'ies'
    return w + 's'

def make_pairs(words):
    out, seen = [], set()
    for w in words:
        if w in seen: continue
        seen.add(w)
        pl = pluralize(w)
        if pl != w:
            out.append({'sg': w, 'pl': pl})
    return out

_SUBJ_SG = [
 'duck','dog','cat','bird','horse','fox','lion','frog','goat','bear','wolf','owl','seal','cow','pig',
 'hen','rat','crab','snail','moth','bee','ant','toad','hawk','crow','mole','newt','wasp','goose','mouse',
 'calf','lamb','colt','chick','cub','pup','kit','doe','ewe','ram','bull','mare','sow','drake','finch',
 'wren','lark','dove','gull','swan','heron','robin','sparrow','beetle','spider','lizard','otter','badger',
 'rabbit','hare','stoat','vole','shrew','weasel','ferret','mink','marten','raccoon','skunk','beaver',
 'squirrel','chipmunk','gopher','hamster','gerbil','hedgehog','porcupine','possum','koala','wombat','camel',
 'llama','donkey','mule','pony','zebra','antelope','gazelle','buffalo','bison','elk','panda','tiger',
 'leopard','cheetah','jaguar','cougar','lynx','bobcat','jackal','coyote','dingo','hyena','meerkat',
 'parrot','falcon','eagle','osprey','magpie','starling','pigeon','quail','pheasant','turkey','peacock',
 'flamingo','pelican','puffin','penguin','turtle','tortoise','gecko','iguana','salamander','minnow',
 'mantis','cricket','locust','hornet','weevil','aphid','earwig','firefly','ladybug',
 'dragonfly','grasshopper','caterpillar','centipede','millipede','tick','flea','gnat','midge','termite',
 'clam','oyster','mussel','prawn','shrimp','lobster','barnacle','urchin','jellyfish','starfish',
 'eel','herring','sardine','shark','dolphin','whale','walrus','manatee','seahorse','stingray','octopus',
 'squid','cuttlefish','cobra','viper','python','adder','mamba','boa','rattler','skink','chameleon','monitor',
 'crane','stork','ibis','egret','plover','sandpiper','warbler','thrush','bunting',
 'kestrel','harrier','buzzard','vulture','condor','raven','jackdaw','rook','jay','nuthatch',
 'mongoose','aardvark','pangolin','tapir','okapi','ibex','chamois','markhor','tahr',
 'wallaby','quokka','bandicoot','numbat','dunnart','quoll','potoroo','bilby','bettong',
 'ox','ape','yak','bat','ray','boar','stag','deer','fawn','foal','joey','tern','kite',
 'slug','worm','grub','larva','moose','sloth','lemur','hippo','rhino','sheep','hound','puppy',
 'piglet','kitten','cock','stallion','filly','heifer','steer','wether','hog',
 'gander','gosling','cygnet','duckling','fledgling','nestling','tadpole','guppy','koi',
]
SUBJECTS = make_pairs(_SUBJ_SG)

_DIST_SG = [
 'tree','car','box','wall','table','house','pond','bridge','gate','fence','rock','lamp','chair','shed',
 'barn','cart','bench','post','window','door','crate','basket','barrel','wagon','ladder','pillar','statue',
 'hedge','well','stove','cabinet','desk','sofa','stool','mirror','clock','vase','plant','bush','stump',
 'log','boulder','fountain','column','arch','tower','cabin','tent','cottage','garage','porch','pole','sign',
 'mailbox','trough','crib','kennel','coop','hive','nest','burrow',
 'lantern','bucket','trellis','planter','urn','birdbath','sundial','gazebo','pergola','archway',
 'wheelbarrow','toolshed','greenhouse','beehive','scarecrow','flagpole','signpost','milestone','culvert','drainpipe',
 'cistern','aqueduct','footbridge','stile','turnstile','railing','banister','balustrade','parapet','buttress',
 'chimney','rooftop','awning','canopy','veranda','balcony','staircase','doorway','threshold','alcove',
 'pallet','drum','sack','bin','vat','tub','jug','pail','keg','plank','beam',
 'rafter','joist','girder','strut','brace','truss','panel','board','curb','ledge','sill',
 'mantel','shelf','rack','hook','peg','knob','latch','hinge','bolt','nail','screw',
 'clamp','vice','anvil','forge','kiln','furnace','boiler','tank','pipe','valve','gauge',
 'dial','switch','socket','cable','wire','rope','chain','cord','strap','belt','buckle',
 'clasp','gravel','mound','heap','pile','mulch','compost','bale','haystack','silo','granary',
 'stable','paddock','pen','corral','pasture','meadow','orchard','vineyard','grove','thicket','bramble',
 'bracken','fern','reed','rush','sedge','moss','lichen','vine','ivy','creeper','shrub',
 'sapling','seedling','bulb','tuber','root','crag','cliff','ridge','slope','bank','dune',
 'mesa','butte','gorge','ravine','gully','ditch','furrow','trench','moat','embankment','levee',
 'dyke','weir','sluice','lock','dam','reservoir','crockery','platter','saucer','kettle','cauldron',
 'pot','pan','skillet','griddle','ladle','whisk','sieve','colander','funnel','jar','flask',
 'vial','beaker','tumbler','goblet','chalice',
]
DISTRACTORS = make_pairs(_DIST_SG)

PREPS = ["near the", "behind the", "beside the", "past the",
         "under the", "by the", "above the", "below the",
         "beneath the", "around the", "between the", "atop the",
         "outside the", "opposite the", "alongside the"]

_VERB = [
    ("is", "are"), ("was", "were"), ("has", "have"), ("does", "do"),
    ("goes", "go"), ("seems", "seem"), ("looks", "look"), ("feels", "feel"),
    ("stays", "stay"), ("grows", "grow"), ("knows", "know"), ("comes", "come"),
    ("runs", "run"), ("jumps", "jump"), ("sleeps", "sleep"), ("eats", "eat"),
    ("walks", "walk"), ("climbs", "climb"), ("hides", "hide"), ("rests", "rest"),
    ("plays", "play"), ("waits", "wait"), ("moves", "move"), ("watches", "watch"),
    ("carries", "carry"), ("gathers", "gather"), ("scatters", "scatter"), ("glows", "glow"),

    ("crawls", "crawl"), ("hops", "hop"), ("flies", "fly"), ("swims", "swim"),
    ("dives", "dive"), ("glides", "glide"), ("trots", "trot"), ("gallops", "gallop"),
    ("paces", "pace"), ("wanders", "wander"), ("roams", "roam"), ("wanders", "wander"),
    ("scurries", "scurry"), ("dashes", "dash"), ("bounds", "bound"), ("leaps", "leap"),
    ("strolls", "stroll"), ("creeps", "creep"), ("crouches", "crouch"), ("stands", "stand"),
    ("sits", "sit"), ("lies", "lie"), ("sniffs", "sniff"), ("licks", "lick"),
    ("bites", "bite"), ("chews", "chew"), ("drinks", "drink"), ("hunts", "hunt"),
    ("tracks", "track"), ("chases", "chase"), ("follows", "follow"), ("escapes", "escape"),
    ("avoids", "avoid"), ("spots", "spot"), ("finds", "find"), ("searches", "search"),
    ("explores", "explore"), ("investigates", "investigate"), ("peeks", "peek"), ("peers", "peer"),
    ("listens", "listen"), ("hears", "hear"), ("calls", "call"), ("chirps", "chirp"),
    ("sings", "sing"), ("howls", "howl"), ("growls", "growl"), ("barks", "bark"),
    ("roars", "roar"), ("hisses", "hiss"), ("squeaks", "squeak"), ("buzzes", "buzz"),
    ("croaks", "croak"), ("chatters", "chatter"), ("blinks", "blink"), ("stares", "stare"),
    ("wags", "wag"), ("wags", "wag"), ("wags", "wag"), ("waves", "wave"),
    ("shakes", "shake"), ("stretches", "stretch"), ("scratches", "scratch"), ("rolls", "roll"),
    ("spins", "spin"), ("balances", "balance"), ("perches", "perch"), ("nests", "nest"),
    ("digs", "dig"), ("burrows", "burrow"), ("builds", "build"), ("forages", "forage"),
    ("collects", "collect"), ("drops", "drop"), ("pushes", "push"), ("pulls", "pull"),
    ("nudges", "nudge"), ("tugs", "tug"), ("grabs", "grab"), ("holds", "hold"),
    ("reaches", "reach"), ("touches", "touch"), ("pounces", "pounce"), ("catches", "catch"),
    ("scrambles", "scramble"), ("floats", "float"), ("drifts", "drift"), ("lingers", "linger"),
    ("appears", "appear"), ("vanishes", "vanish"), ("returns", "return"), ("approaches", "approach"),
    ("leaves", "leave"), ("crosses", "cross"), ("circles", "circle"), ("faces", "face"),
        ("charges", "charge"), ("retreats", "retreat"), ("advances", "advance"), ("flees", "flee"),
    ("dodges", "dodge"), ("weaves", "weave"), ("sloshes", "slosh"), ("splashes", "splash"),
    ("paddles", "paddle"), ("skims", "skim"), ("soars", "soar"), ("hovers", "hover"),
    ("swoops", "swoop"), ("glimpses", "glimpse"), ("observes", "observe"), ("notices", "notice"),
    ("recognizes", "recognize"), ("ignores", "ignore"), ("encounters", "encounter"), ("meets", "meet"),
    ("greets", "greet"), ("nuzzles", "nuzzle"), ("grooms", "groom"), ("preens", "preen"),
    ("bathes", "bathe"), ("shivers", "shiver"), ("pants", "pant"), ("breathes", "breathe"),
    ("yawns", "yawn"), ("dozes", "doze"), ("awakens", "awaken"), ("awakens", "awaken"),
    ("awakens", "awaken"), ("dreams", "dream"), ("relaxes", "relax"), ("freezes", "freeze"),
    ("hesitates", "hesitate"), ("pauses", "pause"), ("continues", "continue"), ("rushes", "rush"),
    ("lingers", "linger"), ("settles", "settle"), ("emerges", "emerge"), ("disappears", "disappear"),
    ("enters", "enter"), ("exits", "exit"), ("descends", "descend"), ("ascends", "ascend"),
    ("circulates", "circulate"), ("patrols", "patrol"), ("guards", "guard"), ("defends", "defend"),
    ("attacks", "attack"), ("threatens", "threaten"), ("confronts", "confront"), ("challenges", "challenge"),
    ("snaps", "snap"), ("strikes", "strike"), ("swats", "swat"), ("pokes", "poke"),
    ("prods", "prod"), ("taps", "tap"), ("presses", "press"), ("squeezes", "squeeze"),
    ("lifts", "lift"), ("raises", "raise"), ("lowers", "lower"), ("drags", "drag"),
    ("hauls", "haul"), ("tosses", "toss"), ("flings", "fling"), ("flips", "flip"),
    ("spills", "spill"), ("spreads", "spread"), ("piles", "pile"), ("stacks", "stack"),
    ("cracks", "crack"), ("breaks", "break"), ("tears", "tear"), ("rips", "rip"),
    ("sniffs", "sniff"), ("smells", "smell"), ("tastes", "taste"), ("gazes", "gaze"),
    ("glances", "glance"), ("examines", "examine"), ("inspects", "inspect"), ("monitors", "monitor"),
    ("signals", "signal"), ("beckons", "beckon"), ("responds", "respond"), ("reacts", "react"),
    ("competes", "compete"), ("cooperates", "cooperate"), ("shares", "share"), ("protects", "protect"),
    ("comforts", "comfort"), ("surrounds", "surround"), ("faces", "face"), ("turns", "turn"),
    ("ambles", "amble"), ("ambles", "amble"), ("stomps", "stomp"), ("trudges", "trudge"),
("tiptoes", "tiptoe"), ("sidles", "sidle"), ("saunters", "saunter"), ("meanders", "meander"),
("sloshes", "slosh"), ("sloshes", "slosh"), ("wades", "wade"), ("sloshes", "slosh"),
("traverses", "traverse"), ("threads", "thread"), ("zigzags", "zigzag"), ("zips", "zip"),
("zooms", "zoom"), ("scales", "scale"), ("vaults", "vault"), ("vaults", "vault"),
("lunges", "lunge"), ("charges", "charge"), ("surges", "surge"), ("retreats", "retreat"),
("backs", "back"), ("sidesteps", "sidestep"), ("dodges", "dodge"), ("ducks", "duck"),
("weaves", "weave"), ("bobs", "bob"), ("bounces", "bounce"), ("tumbles", "tumble"),
("topples", "topple"), ("stumbles", "stumble"), ("slips", "slip"), ("slides", "slide"),
("skids", "skid"), ("skitters", "skitter"), ("scampers", "scamper"), ("romps", "romp"),
("frolics", "frolic"), ("capers", "caper"), ("prances", "prance"), ("strides", "stride"),
("marches", "march"), ("parades", "parade"), ("loops", "loop"), ("spirals", "spiral"),
("twirls", "twirl"), ("twists", "twist"), ("turns", "turn"), ("veers", "veer"),
("drifts", "drift"), ("coasts", "coast"), ("sails", "sail"), ("rafts", "raft"),
("paddles", "paddle"), ("rows", "row"), ("coils", "coil"), ("uncoils", "uncoil"),
("wriggles", "wriggle"), ("wiggles", "wiggle"), ("squirms", "squirm"), ("thrashes", "thrash"),
("flaps", "flap"), ("flutters", "flutter"), ("beats", "beat"), ("fans", "fan"),
("perches", "perch"), ("alights", "alight"), ("lands", "land"), ("takes off", "take off"),
("circles", "circle"), ("orbits", "orbit"), ("cruises", "cruise"), ("soars", "soar"),
("hovers", "hover"), ("swoops", "swoop"), ("banks", "bank"), ("dives", "dive"),
("plunges", "plunge"), ("surfaces", "surface"), ("floats", "float"), ("basks", "bask"),
("sunbathes", "sunbathe"), ("warms", "warm"), ("cools", "cool"), ("shivers", "shiver"),
("pants", "pant"), ("gasps", "gasp"), ("snorts", "snort"), ("snuffles", "snuffle"),
("sneezes", "sneeze"), ("coughs", "cough"), ("sighs", "sigh"), ("blows", "blow"),
("licks", "lick"), ("laps", "lap"), ("gulps", "gulp"), ("nibbles", "nibble"),
("gnaws", "gnaw"), ("munches", "munch"), ("swallows", "swallow"), ("regurgitates", "regurgitate"),
("forages", "forage"), ("grazes", "graze"), ("browses", "browse"), ("harvests", "harvest"),
("plucks", "pluck"), ("picks", "pick"), ("collects", "collect"), ("stores", "store"),
("hoards", "hoard"), ("buries", "bury"), ("retrieves", "retrieve"), ("uncovers", "uncover"),
("uncovers", "uncover"), ("excavates", "excavate"), ("scrapes", "scrape"), ("rakes", "rake"),
("probes", "probe"), ("roots", "root"), ("rummages", "rummage"), ("sniffs out", "sniff out"),
("tracks", "track"), ("stalks", "stalk"), ("pursues", "pursue"), ("ambushes", "ambush"),
("captures", "capture"), ("seizes", "seize"), ("grips", "grip"), ("clutches", "clutch"),
("pins", "pin"), ("releases", "release"), ("abandons", "abandon"), ("escorts", "escort"),
("follows", "follow"), ("leads", "lead"), ("guides", "guide"), ("herds", "herd"),
("joins", "join"), ("separates", "separate"), ("clusters", "cluster"), ("gathers", "gather"),
("crowds", "crowd"), ("surrounds", "surround"), ("approaches", "approach"), ("withdraws", "withdraw"),
("greets", "greet"), ("acknowledges", "acknowledge"), ("alerts", "alert"), ("warns", "warn"),
("signals", "signal"), ("beckons", "beckon"), ("answers", "answer"), ("responds", "respond"),
("communicates", "communicate"), ("chatters", "chatter"), ("trills", "trill"), ("warbles", "warble"),
("coos", "coo"), ("cackles", "cackle"), ("yelps", "yelp"), ("whimpers", "whimper"),
("whines", "whine"), ("moos", "moo"), ("bleats", "bleat"), ("neighs", "neigh"),
("brays", "bray"), ("trumpets", "trumpet"), ("bellows", "bellow"), ("snarls", "snarl"),
("snaps", "snap"), ("bares", "bare"), ("displays", "display"), ("poses", "pose"),
("postures", "posture"), ("preens", "preen"), ("grooms", "groom"), ("scrubs", "scrub"),
("cleans", "clean"), ("rubs", "rub"), ("brushes", "brush"), ("combs", "comb"),
("stretches", "stretch"), ("flexes", "flex"), ("curls", "curl"), ("uncurls", "uncurl"),
("settles", "settle"), ("nestles", "nestle"), ("snuggles", "snuggle"), ("huddles", "huddle"),
("dozes", "doze"), ("awakens", "awaken"), ("rests", "rest"), ("recovers", "recover")
]


VERBS = [{'sg':a,'pl':b} for a,b in dict.fromkeys(_VERB)]

COPULAS = [{'sg':'is','pl':'are'}]

ADJECTIVES = [
    "small", "fuzzy", "curious", "quiet", "clever", "tiny", "strange", "gentle",
    "bright", "sleepy", "nervous", "calm", "round", "spotted", "shy", "bold",
    "scruffy", "slender", "plump", "wary", "restless", "timid", "lively", "drowsy",
    "skittish", "sturdy", "graceful", "clumsy", "fierce", "mellow", "jittery", "placid",
]

ATTRACTORS = [0, 1, 2, 3]
N_FRAMES   = 44

# ============================================================
#  Tokenization filter FIRST, then GLOBAL 50/50 split OVER PAIRS.
#  sg/pl banks (subjects, verbs, distractors) split as whole LEMMAS
#  keyed by sg, so a word's sg & pl always land on the same side and
#  a number-pair can appear only in train OR test. Adjectives and
#  prep heads split per token. Copula (is/are) is shared scaffolding,
#  folded into the verb condition (target_type='verb').
# ============================================================
rng = random.Random(0)

def _tokenizer(lm):
    candidates = [
        getattr(lm, "tokenizer", None),
        getattr(getattr(lm, "processor", None), "tokenizer", None),
        getattr(getattr(lm, "tokenizer", None), "tokenizer", None),
    ]
    for c in candidates:
        if c is not None and hasattr(c, "encode"):
            return c
    raise AttributeError("Could not find a tokenizer with .encode on lm")

TOK = _tokenizer(lm)
print(type(TOK).__name__, "| has encode:", hasattr(TOK, "encode"))

def n_tokens(word, leading_space=True):
    s = (" " + word) if leading_space else word
    return len(TOK.encode(s, add_special_tokens=False))

def is_single_token(word):
    return n_tokens(word) == 1

def filter_pairs(bank):
    return [d for d in bank if is_single_token(d["sg"]) and is_single_token(d["pl"])]

VERBS       = filter_pairs(VERBS)
COPULAS     = filter_pairs(COPULAS)
SUBJECTS    = filter_pairs(SUBJECTS)
DISTRACTORS = filter_pairs(DISTRACTORS)
ADJECTIVES  = [a for a in ADJECTIVES if is_single_token(a)]
print(f"after single-token filter: {len(VERBS)} verbs, {len(COPULAS)} copulas (folded into verb), "
      f"{len(SUBJECTS)} subjects, {len(DISTRACTORS)} distractors, {len(ADJECTIVES)} adjectives")

SHARED_FUNC = {'the', 'is', 'are'}

def prep_head(p):
    toks = [t for t in re.findall(r"[A-Za-z]+", p.lower()) if t not in SHARED_FUNC]
    return toks[0] if toks else p.lower()

# ---- union-find split over content strings (pairs co-travel) ----
PAIR_BANKS = {'subj': SUBJECTS, 'verb': VERBS, 'dist': DISTRACTORS}

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb

all_strings = set()
for bank in PAIR_BANKS.values():
    for d in bank:
        union(d['sg'], d['pl'])
        all_strings.update([d['sg'], d['pl']])
for w in ADJECTIVES:
    find(w); all_strings.add(w)
for p in PREPS:
    h = prep_head(p); find(h); all_strings.add(h)

comps = {}
for s in all_strings:
    comps.setdefault(find(s), set()).add(s)
units = [tuple(sorted(c)) for c in comps.values()]
units.sort()
rng.shuffle(units)
mid = len(units) // 2
TRAIN_VOCAB, TEST_VOCAB = set(), set()
for i, comp in enumerate(units):
    (TRAIN_VOCAB if i < mid else TEST_VOCAB).update(comp)
print(f"split units (string components): {len(units)}  ->  "
      f"train-strings {len(TRAIN_VOCAB)} / test-strings {len(TEST_VOCAB)}")

def split_pairs(bank):
    return ([d for d in bank if d['sg'] in TRAIN_VOCAB],
            [d for d in bank if d['sg'] in TEST_VOCAB])
def split_words(words):
    return ([w for w in words if w in TRAIN_VOCAB],
            [w for w in words if w in TEST_VOCAB])
def split_preps(preps):
    return ([p for p in preps if prep_head(p) in TRAIN_VOCAB],
            [p for p in preps if prep_head(p) in TEST_VOCAB])

SUBJ_TRAIN, SUBJ_TEST = split_pairs(SUBJECTS)
VERB_TRAIN, VERB_TEST = split_pairs(VERBS)
DIST_TRAIN, DIST_TEST = split_pairs(DISTRACTORS)
ADJ_TRAIN,  ADJ_TEST  = split_words(ADJECTIVES)
PREP_TRAIN, PREP_TEST = split_preps(PREPS)

print("per-bank train/test sizes:")
for name, tr, te in [("subjects",SUBJ_TRAIN,SUBJ_TEST),("verbs",VERB_TRAIN,VERB_TEST),
                     ("dists",DIST_TRAIN,DIST_TEST),("adjs",ADJ_TRAIN,ADJ_TEST),
                     ("preps",PREP_TRAIN,PREP_TEST)]:
    print(f"  {name:9s}: {len(tr)} / {len(te)}")

def forms(banks_pairs, banks_words, preps):
    s = set()
    for b in banks_pairs:
        for d in b: s.update([d['sg'], d['pl']])
    for b in banks_words:
        s.update(b)
    for p in preps: s.add(prep_head(p))
    return s
tr_forms = forms([SUBJ_TRAIN,VERB_TRAIN,DIST_TRAIN],[ADJ_TRAIN],PREP_TRAIN)
te_forms = forms([SUBJ_TEST, VERB_TEST, DIST_TEST ],[ADJ_TEST ],PREP_TEST)
print("content-form overlap train∩test:", len(tr_forms & te_forms), sorted(tr_forms & te_forms)[:20])

# ============================================================
#  Generate stimuli — VERB condition only (lexical verb + copula).
#  New template: adjective retained across ALL attractor levels.
#    n=0 : The {adj} {subj} {verb}
#    n>0 : The {adj} {subj} {pp_chain} {verb}
#  Distractors are OPPOSITE number of the target (max interference):
#    sg target -> pl distractors; pl target -> sg distractors.
#  One row per (subject-frame) carries both numbers as four sentences.
# ============================================================
N_PER_COND = 700

def cap(s): return s[0].upper() + s[1:]
def cond(t, n): return f"target_{t}_att{n}_opp"
def mk(*parts):
    return cap(re.sub(r"\s+", " ", " ".join(p for p in parts if p)).strip())
def _toklen(text):
    return len(TOK.encode(text, add_special_tokens=False))

def build_rows(subjects, verbs, adjectives, preps, distractors, split_name):
    def pp_chain(offset, n, dist_num):
        return " ".join(
            f"{preps[(offset+j) % len(preps)]} {distractors[(offset+j) % len(distractors)][dist_num]}"
            for j in range(n)
        )
    rows = []
    for si, subj in enumerate(subjects):
        for n in ATTRACTORS:
            for f in range(N_FRAMES):
                offset = si + f
                dn_sg, dn_pl = 'pl', 'sg'
                pp_sg = pp_chain(offset, n, dn_sg) if n else ""
                pp_pl = pp_chain(offset, n, dn_pl) if n else ""

                # ---------------- VERB (lexical) ----------------
                v   = verbs[(si * 7 + f * 3) % len(verbs)]
                adj = adjectives[(si + f) % len(adjectives)]
                rows.append({
                    'split':split_name,'target_type':'verb','attractors':n,
                    'condition':cond('verb',n),
                    'subject_word_sg': subj['sg'], 'subject_word_pl': subj['pl'],
                    'target_word_sg': v['sg'], 'target_word_pl': v['pl'],
                    'base_sentence_sg': mk("The", adj, subj['sg'], pp_sg),
                    'base_sentence_pl': mk("The", adj, subj['pl'], pp_pl),
                    'good_singular': mk("The", adj, subj['sg'], pp_sg, v['sg']),
                    'bad_singular':  mk("The", adj, subj['sg'], pp_sg, v['pl']),
                    'good_plural':   mk("The", adj, subj['pl'], pp_pl, v['pl']),
                    'bad_plural':    mk("The", adj, subj['pl'], pp_pl, v['sg']),
                })

                # ---------- VERB (copula is/are) — folded into verb ----------
                c     = COPULAS[0]
                adj_c = adjectives[(si * 5 + f * 2 + 7) % len(adjectives)]
                rows.append({
                    'split':split_name,'target_type':'verb','attractors':n,
                    'condition':cond('verb',n),
                    'subject_word_sg': subj['sg'], 'subject_word_pl': subj['pl'],
                    'target_word_sg': c['sg'], 'target_word_pl': c['pl'],
                    'base_sentence_sg': mk("The", adj_c, subj['sg'], pp_sg),
                    'base_sentence_pl': mk("The", adj_c, subj['pl'], pp_pl),
                    'good_singular': mk("The", adj_c, subj['sg'], pp_sg, c['sg']),
                    'bad_singular':  mk("The", adj_c, subj['sg'], pp_sg, c['pl']),
                    'good_plural':   mk("The", adj_c, subj['pl'], pp_pl, c['pl']),
                    'bad_plural':    mk("The", adj_c, subj['pl'], pp_pl, c['sg']),
                })
    return rows

def finalize(rows, n_per_cell):
    df = pd.DataFrame(rows).drop_duplicates(subset=['good_singular']).reset_index(drop=True)
    df['tok_len_sg'] = df['good_singular'].map(_toklen)
    df['tok_len_pl'] = df['good_plural'].map(_toklen)
    keep_masks = []
    for ttype in df['target_type'].unique():
        for n in ATTRACTORS:
            cell_mask = (df['target_type'] == ttype) & (df['attractors'] == n)
            cell = df[cell_mask]
            if len(cell) == 0:
                continue
            mode_sg = cell['tok_len_sg'].mode().iloc[0]
            mode_pl = cell['tok_len_pl'].mode().iloc[0]
            keep_masks.append(cell_mask & (df['tok_len_sg'] == mode_sg)
                                        & (df['tok_len_pl'] == mode_pl))
    uniform = pd.concat([df[m] for m in keep_masks], ignore_index=False)
    pieces = []
    for ttype in uniform['target_type'].unique():
        for n in ATTRACTORS:
            cell = uniform[(uniform['target_type'] == ttype) & (uniform['attractors'] == n)]
            pieces.append(cell.sample(min(len(cell), n_per_cell), random_state=0))
    return pd.concat(pieces, ignore_index=True)

def add_uuid_cols(df):
    """Attach fresh random UUIDs: one set_id per row, plus one id each
    for the singular item and the plural item of that set."""
    n = len(df)
    df['set_id']           = [str(uuid.uuid4()) for _ in range(n)]
    df['singular_item_id'] = [str(uuid.uuid4()) for _ in range(n)]
    df['plural_item_id']   = [str(uuid.uuid4()) for _ in range(n)]
    return df

train_rows = build_rows(SUBJ_TRAIN, VERB_TRAIN, ADJ_TRAIN, PREP_TRAIN, DIST_TRAIN, "train")
test_rows  = build_rows(SUBJ_TEST,  VERB_TEST,  ADJ_TEST,  PREP_TEST,  DIST_TEST,  "test")

bal_train = finalize(train_rows, N_PER_COND)
bal_test  = finalize(test_rows,  N_PER_COND)

bal = pd.concat([bal_train, bal_test], ignore_index=True)
bal.insert(0, 'idx', range(len(bal)))
add_uuid_cols(bal)

print("Grid (rows=condition, cols=attractors) — TRAIN:")
print(bal[bal.split=='train'].pivot_table(index='target_type', columns='attractors',
      values='idx', aggfunc='count', fill_value=0))
print("\nGrid — TEST:")
print(bal[bal.split=='test'].pivot_table(index='target_type', columns='attractors',
      values='idx', aggfunc='count', fill_value=0))
print("\nper split:", dict(bal.groupby('split').size()))
print("TOTAL rows:", len(bal),
      "| unique sg base sentences:", bal['base_sentence_sg'].nunique())

bal.to_csv("agreement_target_natural.csv", index=False)

# ============================================================
#  Wug-subject stimuli: subject -> [wug]/[wugs]. Surrounding real
#  words drawn ONLY from TEST-half pools (disjoint from train).
#  VERB condition only (lexical + copula). split="test".
# ============================================================
WUG_N_FRAMES = 200
WUG = {'sg': '[wug]', 'pl': '[wugs]'}

W_ADJ, W_VERB, W_PREP, W_DIST = ADJ_TEST, VERB_TEST, PREP_TEST, DIST_TEST
N_WUG_SLOTS = max(len(SUBJ_TEST), 30)

def wug_pp_chain(offset, n, dist_num):
    return " ".join(
        f"{W_PREP[(offset+j) % len(W_PREP)]} {W_DIST[(offset+j) % len(W_DIST)][dist_num]}"
        for j in range(n)
    )

rows_wug = []
for si in range(N_WUG_SLOTS):
    for n in ATTRACTORS:
        for f in range(WUG_N_FRAMES):
            offset = si + f
            dn_sg, dn_pl = 'pl', 'sg'
            pp_sg = wug_pp_chain(offset, n, dn_sg) if n else ""
            pp_pl = wug_pp_chain(offset, n, dn_pl) if n else ""
            ai  = (si * 5 + f * 2) % len(W_ADJ)
            aci = (si * 5 + f * 2 + 7) % len(W_ADJ)
            vi  = (si * 7 + f * 3) % len(W_VERB)

            # ---------------- VERB (lexical) ----------------
            v   = W_VERB[vi]
            adj = W_ADJ[ai]
            rows_wug.append({
                'split':'test','target_type':'verb','attractors':n,
                'condition':cond('verb',n),
                'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                'target_word_sg': v['sg'], 'target_word_pl': v['pl'],
                'base_sentence_sg': mk("The", adj, WUG['sg'], pp_sg),
                'base_sentence_pl': mk("The", adj, WUG['pl'], pp_pl),
                'good_singular': mk("The", adj, WUG['sg'], pp_sg, v['sg']),
                'bad_singular':  mk("The", adj, WUG['sg'], pp_sg, v['pl']),
                'good_plural':   mk("The", adj, WUG['pl'], pp_pl, v['pl']),
                'bad_plural':    mk("The", adj, WUG['pl'], pp_pl, v['sg']),
            })

            # ---------- VERB (copula is/are) — folded into verb ----------
            c     = COPULAS[0]
            adj_c = W_ADJ[aci]
            rows_wug.append({
                'split':'test','target_type':'verb','attractors':n,
                'condition':cond('verb',n),
                'subject_word_sg': WUG['sg'], 'subject_word_pl': WUG['pl'],
                'target_word_sg': c['sg'], 'target_word_pl': c['pl'],
                'base_sentence_sg': mk("The", adj_c, WUG['sg'], pp_sg),
                'base_sentence_pl': mk("The", adj_c, WUG['pl'], pp_pl),
                'good_singular': mk("The", adj_c, WUG['sg'], pp_sg, c['sg']),
                'bad_singular':  mk("The", adj_c, WUG['sg'], pp_sg, c['pl']),
                'good_plural':   mk("The", adj_c, WUG['pl'], pp_pl, c['pl']),
                'bad_plural':    mk("The", adj_c, WUG['pl'], pp_pl, c['sg']),
            })

bal_wug = finalize(rows_wug, N_PER_COND)
bal_wug.insert(0, 'idx', range(len(bal_wug)))
add_uuid_cols(bal_wug)

# --- sanity: no wug content word appears in the real-word TRAIN set ---
train_forms = set()
for col in ('good_singular', 'good_plural'):
    for s in bal[bal.split == 'train'][col]:
        train_forms.update(re.findall(r"[A-Za-z]+", s.lower()))
FUNC = {'the','is','are'}
wug_forms = set()
for col in ('good_singular', 'good_plural'):
    for s in bal_wug[col]:
        for w in re.findall(r"[A-Za-z]+", s.lower()):
            if w not in FUNC and w not in ('wug','wugs'):
                wug_forms.add(w)
leak = wug_forms & train_forms
print("wug content words leaking into TRAIN:", len(leak), sorted(leak)[:20])

print("\nGrid (rows=condition, cols=attractors) — WUG:")
print(bal_wug.pivot_table(index='target_type', columns='attractors',
      values='idx', aggfunc='count', fill_value=0))
print("\nunique sg base sentences (wug):")
print(bal_wug.groupby('target_type')['base_sentence_sg'].nunique())
print("TOTAL rows:", len(bal_wug),
      "| unique sg base sentences:", bal_wug['base_sentence_sg'].nunique())

bal_wug.to_csv("agreement_target_wug.csv", index=False)

Qwen2Tokenizer | has encode: True
after single-token filter: 180 verbs, 1 copulas (folded into verb), 38 subjects, 88 distractors, 25 adjectives
split units (string components): 338  ->  train-strings 325 / test-strings 312
per-bank train/test sizes:
  subjects : 20 / 18
  verbs    : 99 / 81
  dists    : 41 / 47
  adjs     : 9 / 16
  preps    : 5 / 10
content-form overlap train∩test: 0 []
Grid (rows=condition, cols=attractors) — TRAIN:
attractors     0    1    2    3
target_type                    
verb         700  700  700  700

Grid — TEST:
attractors     0    1    2    3
target_type                    
verb         700  700  700  700

per split: {'test': np.int64(2800), 'train': np.int64(2800)}
TOTAL rows: 5600 | unique sg base sentences: 4581
wug content words leaking into TRAIN: 0 []

Grid (rows=condition, cols=attractors) — WUG:
attractors     0    1    2    3
target_type                    
verb         700  700  700  700

unique sg base sentences (wug):
target_type
verb    197

In [4]:
len(_VERB)

420